# Upload the tar file

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import os

# Extract the archive
!tar -xzf twitter.tar.gz

# Check extracted contents
os.listdir()
os.listdir("twitter")


## Create Graph using the network : nodes, edges around egonode

In [ ]:
import os
import networkx as nx

# Directory containing Twitter ego files
data_dir = "twitter"
ego_networks = {}
ego_features = {}
ego_circles = {}

for filename in os.listdir(data_dir):
    if filename.endswith(".edges"):
        ego_id = filename.split(".")[0]
        print(f"\n🔄 Processing ego {ego_id}")

        # --- File paths
        edge_path = os.path.join(data_dir, f"{ego_id}.edges")
        feat_path = os.path.join(data_dir, f"{ego_id}.feat")
        egofeat_path = os.path.join(data_dir, f"{ego_id}.egofeat")
        circles_path = os.path.join(data_dir, f"{ego_id}.circles")

        # --- Load directed edges (Twitter)
        G = nx.DiGraph()
        with open(edge_path, 'r') as f:
            for line in f:
                u, v = map(int, line.strip().split())
                G.add_edge(u, v)

        # Add ego node and edges from ego to all nodes (as per Twitter semantics)
        ego_node = int(ego_id)
        for node in list(G.nodes):
          G.add_edge(ego_node, node)


        # --- Load features (.feat)
        features = []
        node_ids = []
        if os.path.exists(feat_path):
            with open(feat_path, 'r') as f:
                for line in f:
                    parts = list(map(int, line.strip().split()))
                    node_ids.append(parts[0])  # Assume ID is included or inferred
                    features.append(parts)

        # --- Load ego features (.egofeat)
        ego_feat = []
        if os.path.exists(egofeat_path):
            with open(egofeat_path, 'r') as f:
                line = f.readline()
                if line:
                    ego_feat = list(map(int, line.strip().split()))

        # --- Load circles (.circles)
        circles = []
        if os.path.exists(circles_path):
            with open(circles_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) > 1:
                        circles.append(list(map(int, parts[1:])))
        ego_circles[ego_id] = circles

        # --- Store raw results
        ego_networks[ego_id] = G
        ego_features[ego_id] = {
            "ego": ego_feat,
            "nodes": dict(zip(sorted(n for n in G.nodes if n != ego_node), features))
        }

        print(f"✅ Loaded ego {ego_id}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


Trim Circle Members: check sanity

In [ ]:
# Validate: Trim circle members that aren't in graph
for ego_id, circles in ego_circles.items():
    G = ego_networks[ego_id]
    trimmed = []
    for circle in circles:
        trimmed_circle = [n for n in circle if n in G.nodes]
        trimmed.append(trimmed_circle)
    ego_circles[ego_id] = trimmed

    print(f"✂️ Trimmed circles for ego {ego_id} → {len(trimmed)} circles")


Attach Features to Graph Nodes

In [ ]:
# Attach features to each node
for ego_id, G in ego_networks.items():
    ego_node = int(ego_id)
    feats = ego_features[ego_id]
    # Assign ego
    G.nodes[ego_node]['features'] = feats['ego']
    # Assign alters
    for node, feat_vec in feats['nodes'].items():
        G.nodes[node]['features'] = feat_vec

Show the graph

In [ ]:
pip install matplotlib


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

# Choose an example ego_id (already loaded in ego_networks)
ego_id_to_plot = next(iter(ego_networks))  # or manually set: '7424642'

G = ego_networks[ego_id_to_plot]

# Plot settings
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)  # layout algorithm (can tweak)

# Draw nodes and edges
nx.draw_networkx_nodes(G, pos, node_size=50, node_color="skyblue", alpha=0.7)
nx.draw_networkx_edges(G, pos, alpha=0.5, arrows=True)
nx.draw_networkx_labels(G, pos, labels={int(ego_id_to_plot): "EGO"}, font_size=12, font_color="red")

plt.title(f"Ego Network for Node {ego_id_to_plot}")
plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

# Choose ego_id to plot
ego_id_to_plot = next(iter(ego_networks))
G = ego_networks[ego_id_to_plot]
ego_node = int(ego_id_to_plot)

# Compute node color based on number of features
node_color = []
for node in G.nodes():
    features = G.nodes[node]
    feature_count = len(features)  # number of attributes attached
    node_color.append(feature_count)

# Plotting
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)

# Draw nodes colored by feature richness
nx.draw_networkx_nodes(G, pos, node_size=60, node_color=node_color, cmap=plt.cm.viridis, alpha=0.9)

# Draw edges
nx.draw_networkx_edges(G, pos, alpha=0.4, arrows=True)

# Highlight and label ego node
nx.draw_networkx_labels(G, pos, labels={ego_node: "EGO"}, font_size=12, font_color="red")

plt.title(f"Ego Network for Node {ego_id_to_plot} (Node color = Feature count)")
plt.colorbar(plt.cm.ScalarMappable(cmap=plt.cm.viridis), label="Number of Features")
plt.axis("off")
plt.tight_layout()
plt.show()


# Indegree, outdegree and clustering coefficient of each node in each ego network in a Table

In [ ]:
import pandas as pd
import networkx as nx

node_stats_all = []
network_stats = []

for ego_id, G in ego_networks.items():
    ego_node = int(ego_id)
    is_directed = isinstance(G, nx.DiGraph)

    # Compute per-node stats
    for node in G.nodes():
        in_deg = G.in_degree(node) if is_directed else G.degree(node)
        out_deg = G.out_degree(node) if is_directed else G.degree(node)
        clustering = nx.clustering(G, node)

        node_stats_all.append({
            "ego_id": ego_id,
            "node_id": node,
            "in_degree": in_deg,
            "out_degree": out_deg,
            "clustering_coef": clustering
        })

    # Compute network-level stats
    avg_density = nx.density(G)
    avg_clustering = nx.average_clustering(G.to_undirected())  # force undirected for clustering

    network_stats.append({
        "ego_id": ego_id,
        "avg_density": avg_density,
        "avg_clustering_coef": avg_clustering,
        "num_nodes": G.number_of_nodes(),
        "num_edges": G.number_of_edges()
    })

# Convert to DataFrames
node_stats_df = pd.DataFrame(node_stats_all)
network_stats_df = pd.DataFrame(network_stats)

# Display sample output
print("📌 Sample Node-Level Stats:")
display(node_stats_df.head())

print("\n📌 Network-Level Summary:")
display(network_stats_df.head())


# Feature Mapping and Embedding

In [ ]:
import os

def get_textual_features_per_ego(ego_networks, ego_ids, base_path):
    all_textual_features = {}

    for ego_id in ego_ids:
        ego_id = str(ego_id)
        G = ego_networks.get(ego_id)
        if G is None:
            continue

        # Paths
        featnames_path = os.path.join(base_path, f"{ego_id}.featnames")
        egofeat_path = os.path.join(base_path, f"{ego_id}.egofeat")
        feat_path = os.path.join(base_path, f"{ego_id}.feat")

        # --- Load featnames
        featnames = {}
        if os.path.exists(featnames_path):
            with open(featnames_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        try:
                            idx = int(parts[0])
                            featnames[idx] = " ".join(parts[1:])
                        except ValueError:
                            continue

        # --- Load egofeat
        ego_feat_vector = []
        if os.path.exists(egofeat_path):
            with open(egofeat_path, "r") as f:
                line = f.readline().strip()
                if line:
                    ego_feat_vector = list(map(int, line.split()))

        # --- Load feat matrix
        feat_matrix = []
        if os.path.exists(feat_path):
            with open(feat_path, "r") as f:
                for line in f:
                    feat_matrix.append(list(map(int, line.strip().split())))

        # --- Trim featnames to match feature vector length
        feat_len = len(ego_feat_vector)
        featnames = {k: v for k, v in featnames.items() if k < feat_len}

        textual_features = {}

        # --- Ego node
        ego_active = [featnames[i] for i, val in enumerate(ego_feat_vector) if val == 1 and i in featnames]
        textual_features[int(ego_id)] = " ".join(ego_active) if ego_active else "no features"

        # --- Non-ego nodes
        non_ego_nodes = sorted(n for n in G.nodes if int(n) != int(ego_id))
        if len(non_ego_nodes) != len(feat_matrix):
            min_len = min(len(non_ego_nodes), len(feat_matrix))
            non_ego_nodes = non_ego_nodes[:min_len]
            feat_matrix = feat_matrix[:min_len]

        for node, feat_vec in zip(non_ego_nodes, feat_matrix):
            active_feats = [featnames[i] for i, val in enumerate(feat_vec) if val == 1 and i in featnames]
            textual_features[node] = " ".join(active_feats) if active_feats else "no features"

        all_textual_features[ego_id] = textual_features

    return all_textual_features


In [ ]:
ego_ids = list(ego_networks.keys())  # Reconstruct ego_ids from loaded networks


In [ ]:
textual_feature_map = get_textual_features_per_ego(ego_networks, ego_ids, base_path="twitter")

# Show a few mapped results
ego_id = ego_ids[0]
for node, text in list(textual_feature_map[ego_id].items())[:5]:
    print(f"{node}: {text}")


In [ ]:
!pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the SBERT model
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')


Generate Embeddings for Each Node

In [ ]:
node_embeddings = {}  # {ego_id: {node_id: embedding}}

for ego_id, node_texts in textual_feature_map.items():
    embeddings = {}
    node_ids = list(node_texts.keys())
    text_descriptions = list(node_texts.values())

    # Batch encode the node feature texts
    sbert_vectors = sbert_model.encode(text_descriptions, show_progress_bar=False)

    # Map node_id to its embedding
    for node, emb in zip(node_ids, sbert_vectors):
        embeddings[node] = emb

    node_embeddings[ego_id] = embeddings


In [ ]:
# Example: Show an embedding vector
ego_sample = ego_ids[0]
node_sample = list(node_embeddings[ego_sample].keys())[0]
print(f"Node {node_sample} embedding (dim={len(node_embeddings[ego_sample][node_sample])}):")
print(node_embeddings[ego_sample][node_sample])

In [ ]:
import pandas as pd
import os

output_dir = "embeddings_csv"
os.makedirs(output_dir, exist_ok=True)

for ego_id, nodes in node_embeddings.items():
    rows = []
    for node_id, emb in nodes.items():
        row = {
            "ego_id": ego_id,
            "node_id": node_id
        }
        # Add each dimension as a separate column: dim_0, dim_1, ..., dim_383
        row.update({f"dim_{i}": val for i, val in enumerate(emb)})
        rows.append(row)

    df = pd.DataFrame(rows)
    output_path = os.path.join(output_dir, f"embeddings_{ego_id}.csv")
    df.to_csv(output_path, index=False)
    print(f"✅ Exported: {output_path} ({len(rows)} rows)")


In [ ]:
import zipfile

zip_output_path = "embeddings_csv.zip"

with zipfile.ZipFile(zip_output_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, arcname=os.path.relpath(file_path, output_dir))

print(f"\n📦 Zipped all CSVs to: {zip_output_path}")


In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# List to store results
similarity_records = []

for ego_id in ego_ids:
    ego_node = int(ego_id)
    embeddings = node_embeddings[ego_id]

    # Ensure ego node is in embedding set
    if ego_node not in embeddings:
        continue

    ego_emb = embeddings[ego_node].reshape(1, -1)

    for alter_node, alter_emb in embeddings.items():
        if alter_node == ego_node:
            continue  # Skip self-comparison

        alter_emb = alter_emb.reshape(1, -1)
        sim = cosine_similarity(ego_emb, alter_emb)[0][0]

        similarity_records.append({
            "ego_id": ego_id,
            "alter_node": alter_node,
            "cosine_similarity": sim
        })

# Create DataFrame
similarity_df = pd.DataFrame(similarity_records)

# Show full table (or sample if too large)
pd.set_option("display.max_rows", None)
print("📌 Cosine Similarity Between Ego Nodes and Alters")
display(similarity_df)


In [ ]:
similarity_df.sort_values(by="cosine_similarity", ascending=False, inplace=True)
similarity_df.to_csv("ego_alter_similarity.csv", index=False)


In [ ]:
# Compute average similarity for each ego network
average_similarity_df = similarity_df.groupby("ego_id")["cosine_similarity"].mean().reset_index()

# Rename column for clarity
average_similarity_df.rename(columns={"cosine_similarity": "average_cosine_similarity"}, inplace=True)

# Display the result
print("📌 Average Cosine Similarity for Each Ego Network")
display(average_similarity_df)


In [ ]:
# Calculate min and max values
min_val = average_similarity_df["average_cosine_similarity"].min()
max_val = average_similarity_df["average_cosine_similarity"].max()

print(f"Range of ego-alter similarity: {min_val:.4f} to {max_val:.4f}")


In [ ]:
# Compute median value
median_val = average_similarity_df["average_cosine_similarity"].median()

print(f"Median ego-alter similarity: {median_val:.4f}")


In [ ]:
# Compute IQR bounds
q1 = average_similarity_df["average_cosine_similarity"].quantile(0.25)
q3 = average_similarity_df["average_cosine_similarity"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Add outlier flag column
average_similarity_df["is_outlier"] = (
    (average_similarity_df["average_cosine_similarity"] < lower_bound) |
    (average_similarity_df["average_cosine_similarity"] > upper_bound)
)

# Optional: Display the result
print("📌 Average Cosine Similarity with Outlier Flags")
display(average_similarity_df)


# Leiden Community Detection

In [ ]:
!pip install leidenalg igraph


In [ ]:
import igraph as ig
import leidenalg
import networkx as nx

# Dictionary to store detected communities
ego_communities = {}

for ego_id in ego_ids:
    ego_id = str(ego_id)
    G = ego_networks.get(ego_id)
    if G is None:
        continue

    # Convert to undirected for community detection
    G_undirected = G.to_undirected()

    # Convert to igraph
    ig_graph = ig.Graph.TupleList(G_undirected.edges(), directed=False)

    # Run Leiden
    partition = leidenalg.find_partition(ig_graph, leidenalg.ModularityVertexPartition)

    # Map node indices to actual IDs (from NetworkX)
    node_list = list(G_undirected.nodes())
    node_to_community = {node_list[i]: int(comm) for i, comm in enumerate(partition.membership)}

    # Store
    ego_communities[ego_id] = node_to_community

    print(f"✅ Detected {len(set(partition.membership))} communities for ego {ego_id}")


In [ ]:
import igraph as ig
import leidenalg
import os

ego_communities = {}

for ego_id in ego_ids:
    ego_id = str(ego_id)
    G = ego_networks.get(ego_id)
    if G is None:
        continue

    G_undirected = G.to_undirected()
    ig_graph = ig.Graph.TupleList(G_undirected.edges(), directed=False, vertex_name_attr="name")

    # Run Leiden
    partition = leidenalg.find_partition(ig_graph, leidenalg.ModularityVertexPartition)

    node_list = list(G_undirected.nodes())
    node_to_community = {node_list[i]: int(comm) for i, comm in enumerate(partition.membership)}

    # Load .circles
    circles_path = os.path.join("twitter", f"{ego_id}.circles")
    extra_communities = []
    if os.path.exists(circles_path):
        with open(circles_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) > 1:
                    circle_nodes = list(map(int, parts[1:]))
                    extra_communities.append(circle_nodes)

    current_max_comm = max(node_to_community.values(), default=-1)

    for circle in extra_communities:
        existing_community_ids = [node_to_community.get(n) for n in circle if n in node_to_community]
        unique_comms = set(existing_community_ids)

        if len(unique_comms) > 1 or None in existing_community_ids:
            current_max_comm += 1
            for node in circle:
                if node in G_undirected:
                    node_to_community[node] = current_max_comm
                    if node not in ig_graph.vs["name"]:
                        ig_graph.add_vertex(name=node)

            # Add missing edges between circle members
            for i, node in enumerate(circle):
                for j in range(i + 1, len(circle)):
                    other = circle[j]
                    if node == other:
                        continue

                    try:
                        idx1 = ig_graph.vs.find(name=node).index
                        idx2 = ig_graph.vs.find(name=other).index
                        if not ig_graph.are_adjacent(idx1, idx2):
                            ig_graph.add_edge(idx1, idx2)
                    except ValueError:
                        # One of the nodes wasn't found in the graph
                        continue

    ego_communities[ego_id] = node_to_community
    print(f"✅ Final community count for ego {ego_id}: {len(set(node_to_community.values()))}")


In [ ]:
example_ego = ego_ids[0]
for node, comm_id in list(ego_communities[example_ego].items())[:10]:
    print(f"Node {node} → Community {comm_id}")


In [ ]:
import numpy as np

community_centroids = {}

for ego_id in ego_ids:
    ego_id_str = str(ego_id)
    circles = ego_circles.get(ego_id_str, [])
    embeddings = node_embeddings.get(ego_id_str, {})

    centroids = []
    for circle_index, circle_nodes in enumerate(circles):
        vectors = []

        for node in circle_nodes:
            if node in embeddings:
                vectors.append(embeddings[node])

        # Only compute centroid if there's at least one valid vector
        if vectors:
            centroid = np.mean(vectors, axis=0)
            centroids.append(centroid)
        else:
            centroids.append(None)  # Or np.zeros(...), depending on preference

    community_centroids[ego_id_str] = centroids

print("✅ Computed centroids for all communities.")


In [ ]:
centroid_records = []

for ego_id, centroids in community_centroids.items():
    for i, centroid in enumerate(centroids):
        if centroid is not None:
            centroid_records.append({
                "ego_id": ego_id,
                "circle_index": i,
                "centroid_vector": centroid
            })

centroid_df = pd.DataFrame(centroid_records)
display(centroid_df.head())


In [ ]:
# Check type of node_embeddings
print("Type of node_embeddings:", type(node_embeddings))

# Print keys (ego_ids)
print("Sample keys in node_embeddings:", list(node_embeddings.keys())[:3])

# Check one value
sample_ego_id = list(node_embeddings.keys())[0]
print(f"\nType of node_embeddings['{sample_ego_id}']:", type(node_embeddings[sample_ego_id]))

# Print nested keys if it's a dictionary
if isinstance(node_embeddings[sample_ego_id], dict):
    print("Sample nested keys (node_ids):", list(node_embeddings[sample_ego_id].keys())[:3])

    # Check what the inner value looks like
    sample_node_id = list(node_embeddings[sample_ego_id].keys())[0]
    print("Sample embedding:", node_embeddings[sample_ego_id][sample_node_id][:5])
else:
    # It's a vector directly
    print("Sample embedding (flat):", node_embeddings[sample_ego_id][:5])


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_records = []

for record in centroid_records:
    ego_id = record["ego_id"]  # this is a string
    centroid = record["centroid_vector"]
    circle_index = record["circle_index"]

    # Get the ego's own embedding vector
    ego_vector = node_embeddings.get(ego_id, {}).get(int(ego_id))

    if ego_vector is not None and centroid is not None:
        sim = cosine_similarity([ego_vector], [centroid])[0][0]

        similarity_records.append({
            "ego_id": ego_id,
            "circle_index": circle_index,
            "cosine_similarity": sim
        })

similarity_df = pd.DataFrame(similarity_records)
display(similarity_df.head())


In [ ]:
from collections import defaultdict
import networkx as nx

# Store shortest path lengths
shortest_path_records = []

for ego_id in ego_ids:
    ego_id_str = str(ego_id)
    G = ego_networks.get(ego_id_str)
    circles = ego_circles.get(ego_id_str, [])

    if G is None or not G.has_node(int(ego_id)):
        continue

    for i, circle in enumerate(circles):
        min_path_len = float('inf')

        for node in circle:
            try:
                path_len = nx.shortest_path_length(G, source=int(ego_id), target=int(node))
                if path_len < min_path_len:
                    min_path_len = path_len
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                continue

        # Save result
        if min_path_len != float('inf'):
            shortest_path_records.append({
                "ego_id": ego_id_str,
                "circle_index": i,
                "shortest_path_len": min_path_len
            })

# Create DataFrame
path_df = pd.DataFrame(shortest_path_records)
display(path_df.head())


In [ ]:
# Merge on both ego_id and circle_index
combined_df = pd.merge(
    similarity_df,
    path_df,
    on=["ego_id", "circle_index"],
    how="inner"
)

# Optional: Sort by ego_id then circle_index for grouped visibility
combined_df = combined_df.sort_values(by=["ego_id", "circle_index"])

# Display grouped view
display(combined_df.head(10))


In [ ]:
# Export to CSV
combined_df.to_csv("ego_circle_similarity_and_path.csv", index=False)

print("✅ Exported combined DataFrame to 'ego_circle_similarity_and_path.csv'")


In [ ]:
# Add a column to label communities
combined_df['community_type'] = 'external'  # default

# Find primary (most similar) community for each ego
primary_indices = combined_df.groupby('ego_id')['cosine_similarity'].idxmax()

# Update those rows as primary
combined_df.loc[primary_indices, 'community_type'] = 'primary'

# Optional: Display
display(combined_df.sort_values(['ego_id', 'community_type'], ascending=[True, False]).head(10))


In [ ]:
combined_df.to_csv("labeled_ego_communities.csv", index=False)


# **Echo Escape Potential**

# Linear Decay

In [ ]:
import numpy as np
import pandas as pd
from math import log, exp

# Set alpha for exponential/softmax decay
alpha = 0.7

# Initialize results
escape_records = []

for ego_id, group in combined_df.groupby('ego_id'):
    sims = group['cosine_similarity'].values
    dists = group['shortest_path_len'].values

    # Filter out infinite distances if any
    valid = np.isfinite(dists)
    if not valid.any():
        continue

    sims = sims[valid]
    dists = dists[valid]

    # Weight calculations
    linear_weights = 1 / (1 + dists)
    log_weights = 1 / np.log1p(dists)
    exp_weights = np.exp(-alpha * dists)
    softmax_weights = exp_weights / exp_weights.sum()

    # Normalize all except softmax
    linear_weights /= linear_weights.sum()
    log_weights /= log_weights.sum()
    exp_weights /= exp_weights.sum()

    # Compute weighted averages
    escape_linear = np.dot(sims, linear_weights)
    escape_log = np.dot(sims, log_weights)
    escape_exp = np.dot(sims, exp_weights)
    escape_softmax = np.dot(sims, softmax_weights)

    escape_records.append({
        "ego_id": ego_id,
        "escape_linear": escape_linear,
        "escape_log": escape_log,
        "escape_exp": escape_exp,
        "escape_softmax": escape_softmax
    })

# Create DataFrame
escape_df = pd.DataFrame(escape_records)
display(escape_df.head(10))


In [ ]:
escape_df.to_csv("escape_potential_all_decay_methods.csv", index=False)


In [ ]:
# Keep only ego_id and escape_exp columns
escape_df = escape_df[["ego_id", "escape_exp"]]

# Display the updated DataFrame
display(escape_df.head(10))


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

intercommunity_similarity_records = []

# Group all centroid vectors by ego_id
from collections import defaultdict
centroids_by_ego = defaultdict(list)
for record in centroid_records:
    centroids_by_ego[record["ego_id"]].append((record["circle_index"], record["centroid_vector"]))

# Cosine similarity among centroids
for ego_id, centroids in centroids_by_ego.items():
    # Skip if no valid centroids
    valid_centroids = [(i, vec) for i, vec in centroids if vec is not None]
    if len(valid_centroids) < 2:
        continue

    # Compute ego → community similarities to find primary
    ego_vector = node_embeddings.get(ego_id, {}).get(int(ego_id))
    if ego_vector is None:
        continue

    sims = [(i, cosine_similarity([ego_vector], [vec])[0][0]) for i, vec in valid_centroids]
    sims.sort(key=lambda x: x[1], reverse=True)
    primary_index = sims[0][0]
    primary_vector = [vec for i, vec in valid_centroids if i == primary_index][0]

    # Compare primary to all others
    external_vectors = [vec for i, vec in valid_centroids if i != primary_index]
    if not external_vectors:
        continue

    inter_sims = cosine_similarity([primary_vector], external_vectors)[0]
    avg_inter_sim = float(np.mean(inter_sims))

    intercommunity_similarity_records.append({
        "ego_id": ego_id,
        "primary_circle_index": primary_index,
        "intercommunity_similarity": avg_inter_sim
    })

# Create DataFrame
inter_sim_df = pd.DataFrame(intercommunity_similarity_records)
display(inter_sim_df.head())


In [ ]:
# Assuming your DataFrame is called `intercommunity_df`
inter_sim_df.to_csv("intercommunity_similarity.csv", index=False)


In [ ]:
# Group by ego_id and compute average intercommunity similarity
avg_inter_sim_df = inter_sim_df.groupby("ego_id", as_index=False)["intercommunity_similarity"].mean()

# Display the resulting table
display(avg_inter_sim_df.head())


In [ ]:
# Compute range of intercommunity similarity
min_val = avg_inter_sim_df["intercommunity_similarity"].min()
max_val = avg_inter_sim_df["intercommunity_similarity"].max()

print(f"Range of intercommunity_similarity: {min_val:.4f} to {max_val:.4f}")


In [ ]:
# Compute median of average intercommunity similarity
median_val = avg_inter_sim_df["intercommunity_similarity"].median()

print(f"Median intercommunity similarity: {median_val:.4f}")


In [ ]:
# Compute bounds
q1 = avg_inter_sim_df["intercommunity_similarity"].quantile(0.25)
q3 = avg_inter_sim_df["intercommunity_similarity"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Add flag column
avg_inter_sim_df["is_outlier"] = (
    (avg_inter_sim_df["intercommunity_similarity"] < lower_bound) |
    (avg_inter_sim_df["intercommunity_similarity"] > upper_bound)
)


In [ ]:
# Merge average_cosine_similarity with escape potential
final_df = average_similarity_df.merge(escape_df, on='ego_id', how='left')

# Merge with intercommunity similarity
final_df = final_df.merge(avg_inter_sim_df, on='ego_id', how='left')

# Display final summary
print("📊 Final Summary Table")
display(final_df.head(10))


In [ ]:
final_df = final_df.drop(columns=["is_outlier"])
display(final_df.head())


In [ ]:
num_nan_rows = final_df.isna().any(axis=1).sum()
print(f"🔍 Rows with at least one NaN: {num_nan_rows}")


In [ ]:
final_df_dropped = final_df.dropna()
print(f"✅ Dropped rows with NaNs. Remaining rows: {len(final_df_dropped)}")
display(final_df_dropped.head())


In [ ]:
final_df_dropped.to_csv("final_summary_cleaned.csv", index=False)
print("✅ Exported to final_summary_cleaned.csv")


# Clustering Methods

In [ ]:
from sklearn.preprocessing import StandardScaler

# Select features for clustering
features = final_df_dropped[["average_cosine_similarity", "escape_exp", "intercommunity_similarity"]].values

# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)


In [ ]:
from sklearn.cluster import SpectralClustering

# Try different cluster numbers (e.g., 2–10)
n_clusters = 3
spectral = SpectralClustering(n_clusters=n_clusters, affinity='nearest_neighbors', assign_labels='kmeans', random_state=42)
labels = spectral.fit_predict(X_scaled)

# Add cluster labels to the DataFrame
final_df_dropped["cluster"] = labels


In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X_scaled, labels)
print(f"📊 Silhouette Score for Spectral Clustering (k={n_clusters}): {score:.3f}")


In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=n_clusters, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

print("KMeans Silhouette Score:", silhouette_score(X_scaled, kmeans_labels))


In [ ]:
from sklearn.cluster import AgglomerativeClustering

agglo = AgglomerativeClustering(n_clusters=n_clusters)
agglo_labels = agglo.fit_predict(X_scaled)

print("Agglomerative Silhouette Score:", silhouette_score(X_scaled, agglo_labels))


In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

# Filter out noise (-1) before scoring
valid = dbscan_labels != -1
if valid.sum() > 1:
    print("DBSCAN Silhouette Score:", silhouette_score(X_scaled[valid], dbscan_labels[valid]))
else:
    print("DBSCAN didn't form enough valid clusters.")


In [ ]:
from sklearn.cluster import SpectralClustering

# Try different cluster numbers (e.g., 2–10)
n_clusters = 2
spectral = SpectralClustering(n_clusters=n_clusters, affinity='nearest_neighbors', assign_labels='kmeans', random_state=42)
labels = spectral.fit_predict(X_scaled)

# Add cluster labels to the DataFrame
final_df_dropped["cluster"] = labels


from sklearn.metrics import silhouette_score

score = silhouette_score(X_scaled, labels)
print(f"📊 Silhouette Score for Spectral Clustering (k={n_clusters}): {score:.3f}")


In [ ]:
!pip install scikit-learn-extra

from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score

# Choose number of clusters
n_clusters = 3

# Fit KMedoids
kmedoids = KMedoids(n_clusters=n_clusters, random_state=42)
kmedoids_labels = kmedoids.fit_predict(X_scaled)

# Evaluate
kmedoids_score = silhouette_score(X_scaled, kmedoids_labels)
print(f"KMedoids Silhouette Score (k={n_clusters}): {kmedoids_score:.3f}")


In [ ]:
import hdbscan

# Fit HDBSCAN (min_cluster_size affects granularity)
hdb = hdbscan.HDBSCAN(min_cluster_size=5)
hdb_labels = hdb.fit_predict(X_scaled)

# Filter out noise (-1) for silhouette score
valid = hdb_labels != -1
if valid.sum() > 1:
    hdb_score = silhouette_score(X_scaled[valid], hdb_labels[valid])
    print(f"HDBSCAN Silhouette Score (excluding noise): {hdb_score:.3f}")
else:
    print("HDBSCAN did not form enough valid clusters.")


In [ ]:
from sklearn.mixture import GaussianMixture

# Choose number of components
n_components = 2

# Fit GMM
gmm = GaussianMixture(n_components=n_components, random_state=42)
gmm_labels = gmm.fit_predict(X_scaled)

# Evaluate
gmm_score = silhouette_score(X_scaled, gmm_labels)
print(f"GMM Silhouette Score (k={n_components}): {gmm_score:.3f}")


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import numpy as np

# Assume your feature columns were scaled into X_scaled
# And gmm_labels contains the cluster assignments

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot
scatter = ax.scatter(
    X_scaled[:, 0], X_scaled[:, 1], X_scaled[:, 2],
    c=gmm_labels, cmap='viridis', s=40, alpha=0.7
)

# Label axes according to your original features
ax.set_xlabel('Average Cosine Similarity')
ax.set_ylabel('Escape Potential')
ax.set_zlabel('Intercommunity Similarity')
ax.set_title('3D GMM Clustering (Original 3 Features)')

# Legend via colorbar
cbar = plt.colorbar(scatter, ax=ax, pad=0.1)
cbar.set_label('Cluster Label')

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.cluster import OPTICS
from sklearn.metrics import silhouette_score

# Fit OPTICS
optics = OPTICS(min_samples=5, xi=0.05, min_cluster_size=0.05)
optics_labels = optics.fit_predict(X_scaled)

# Evaluate: filter out noise (-1)
valid = optics_labels != -1
if valid.sum() > 1:
    optics_score = silhouette_score(X_scaled[valid], optics_labels[valid])
    print(f"OPTICS Silhouette Score (excluding noise): {optics_score:.3f}")
else:
    print("OPTICS did not form enough valid clusters.")


In [ ]:
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# 1. Extract feature matrix from your final_df_dropped
X = final_df_dropped[["average_cosine_similarity", "escape_exp", "intercommunity_similarity"]].values

# 2. Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. PCA to reduce dimensionality
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# 4. GMM clustering on PCA-reduced data
n_components = 2
gmm = GaussianMixture(n_components=n_components, random_state=42)
gmm_labels = gmm.fit_predict(X_pca)

# 5. Silhouette score (evaluates clustering quality)
gmm_score = silhouette_score(X_pca, gmm_labels)
print(f"GMM Silhouette Score (PCA + k={n_components}): {gmm_score:.3f}")


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=gmm_labels, cmap='viridis', s=60)
plt.title(f"GMM Clustering on PCA-Reduced Data (k={n_components})")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.colorbar(label="Cluster")
plt.grid(True)
plt.show()


In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

# Reduce to 3 components for visualization
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)

# Plot clusters in 3D
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_pca_3d[:, 0], X_pca_3d[:, 1], X_pca_3d[:, 2], c=gmm_labels, cmap='viridis', s=50)
ax.set_title("3D PCA Clustering Visualization")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
plt.show()


In [ ]:
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

# Step 1: Apply PCA with 3 components
pca = PCA(n_components=3)
X_pca_3d = pca.fit_transform(X_scaled)

# Step 2: Fit GMM
gmm = GaussianMixture(n_components=2, random_state=42)
gmm_labels_pca3 = gmm.fit_predict(X_pca_3d)

# Step 3: Evaluate with silhouette score
gmm_score_pca3 = silhouette_score(X_pca_3d, gmm_labels_pca3)
print(f"GMM Silhouette Score (PCA=3, k=2): {gmm_score_pca3:.3f}")
